# Step 15 — Failure-case analysis

We inspect where the simple models and signals failed on the held-out June 7–November 22, 2024 period. This is qualitative diagnosis, not another round of model selection.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "aapl_momentum_sentiment.csv"
HEADLINES_PATH = PROJECT_ROOT / "data" / "processed" / "apple_headlines_finbert.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "failure_cases.csv"
data = pd.read_csv(DATA_PATH, parse_dates=["date"], index_col="date").sort_index().loc[:"2024-11-22"]
headlines = pd.read_csv(HEADLINES_PATH, parse_dates=["trading_date"])

In [2]:
test_size = int(len(data) * 0.25)
test_start = len(data) - test_size
train = data.iloc[: test_start - 5].copy()
test = data.iloc[test_start:].copy()
y_train = train["outperformed"].astype(int)

model_specs = {
    "momentum": ["aapl_momentum_5d"],
    "tone": ["average_sentiment_5d"],
}
analysis = test.copy()
for name, features in model_specs.items():
    model = make_pipeline(StandardScaler(), LogisticRegression(random_state=42))
    model.fit(train[features], y_train)
    analysis[f"{name}_probability"] = model.predict_proba(test[features])[:, 1]
    analysis[f"{name}_prediction"] = model.predict(test[features])
    analysis[f"{name}_correct"] = analysis[f"{name}_prediction"].eq(analysis["outperformed"])

analysis[["outperformed", "momentum_probability", "tone_probability"]].head()

,outperformed,momentum_probability,tone_probability
date,,,
2024-06-07,1,0.515866,0.478921
2024-06-10,1,0.534305,0.478260
2024-06-11,1,0.489110,0.481053
2024-06-12,0,0.475049,0.476284
2024-06-13,0,0.466234,0.481180


## Attach headline context

For each selected date, we attach up to three headlines from its five-trading-day feature window, prioritizing the most strongly positive or negative FinBERT scores.

In [3]:
trading_dates = pd.DatetimeIndex(data.index)

def headline_context(prediction_date, limit=3):
    position = trading_dates.get_loc(prediction_date)
    window_dates = trading_dates[max(0, position - 4) : position + 1]
    recent = headlines.loc[headlines["trading_date"].isin(window_dates)].copy()
    recent["absolute_score"] = recent["sentiment_score"].abs()
    recent = recent.nlargest(limit, "absolute_score")
    return " | ".join(
        f"{row.title} [{row.sentiment_score:+.2f}]" for row in recent.itertuples()
    )

def select_cases(frame, category, count=3, ascending=False, sort_column=None):
    selected = frame.sort_values(sort_column, ascending=ascending).head(count).copy()
    selected["failure_category"] = category
    return selected

## Failure categories

- **Highest-probability tone false positive:** tone predicted outperformance, but AAPL underperformed.
- **Lowest-probability tone false negative:** tone predicted no outperformance, but AAPL outperformed.
- **Contrarian signal failure:** strongly negative tone was followed by underperformance rather than recovery.
- **Positive-tone disappointment:** strongly positive tone was followed by underperformance.
- **Momentum reversal:** strong positive momentum was followed by underperformance.
- **Model disagreement:** momentum and tone produced different classifications.

In [4]:
case_groups = []
case_groups.append(select_cases(
    analysis.loc[analysis["tone_prediction"].eq(1) & analysis["outperformed"].eq(0)],
    "Highest-probability tone false positive", sort_column="tone_probability"
))
case_groups.append(select_cases(
    analysis.loc[analysis["tone_prediction"].eq(0) & analysis["outperformed"].eq(1)],
    "Lowest-probability tone false negative", ascending=True, sort_column="tone_probability"
))
case_groups.append(select_cases(
    analysis.loc[analysis["outperformed"].eq(0)],
    "Contrarian signal failure", ascending=True, sort_column="average_sentiment_5d"
))
case_groups.append(select_cases(
    analysis.loc[analysis["outperformed"].eq(0)],
    "Positive-tone disappointment", sort_column="average_sentiment_5d"
))
case_groups.append(select_cases(
    analysis.loc[analysis["outperformed"].eq(0)],
    "Momentum reversal", sort_column="aapl_momentum_5d"
))
disagreements = analysis.loc[analysis["momentum_prediction"].ne(analysis["tone_prediction"])].copy()
disagreements["probability_gap"] = (
    disagreements["momentum_probability"] - disagreements["tone_probability"]
).abs()
case_groups.append(select_cases(
    disagreements, "Model disagreement", sort_column="probability_gap"
))

cases = pd.concat(case_groups)
cases["headline_context"] = [headline_context(date) for date in cases.index]
cases.index.name = "date"

In [5]:
output_columns = [
    "failure_category", "aapl_momentum_5d", "average_sentiment_5d",
    "headline_count_5d", "aapl_future_return_5d", "spy_future_return_5d",
    "outperformed", "momentum_probability", "momentum_prediction",
    "tone_probability", "tone_prediction", "headline_context",
]
cases[output_columns].to_csv(OUTPUT_PATH)
print(f"Saved {len(cases)} diagnostic cases to {OUTPUT_PATH}")
cases[output_columns]

Saved 18 diagnostic cases to /Users/keishakalra/Desktop/Financial_App/data/processed/failure_cases.csv


,failure_category,aapl_momentum_5d,average_sentiment_5d,headline_count_5d,aapl_future_return_5d,spy_future_return_5d,outperformed,momentum_probability,momentum_prediction,tone_probability,tone_prediction,headline_context
date,,,,,,,,,,,,
2024-10-30,Highest-probability tone false positive,-0.002860,-0.306662,11,-0.032073,0.019017,0,0.533137,1,0.569242,1,Apple stock slips after a big cut for iPhone 1...
2024-10-29,Highest-probability tone false positive,-0.009285,-0.278343,12,-0.043737,-0.008715,0,0.537237,1,0.564559,1,Apple stock slips after a big cut for iPhone 1...
2024-10-31,Highest-probability tone false positive,-0.020211,-0.232233,7,0.006950,0.047429,0,0.544199,1,0.556909,1,Apple Intelligence is here. Early users are un...
2024-11-19,Lowest-probability tone false negative,0.018062,0.250825,3,0.029700,0.017533,1,0.519756,1,0.475835,0,"Smart Home Devices Could Boost Apple Stock, An..."
2024-06-10,Lowest-probability tone false negative,-0.004690,0.236392,44,0.121945,0.021357,1,0.534305,1,0.478260,0,Nvidia closes above $3 trillion for first time...
2024-06-07,Lowest-probability tone false negative,0.024135,0.232461,34,0.079232,0.016423,1,0.515866,1,0.478921,0,Nvidia closes above $3 trillion for first time...
2024-10-30,Contrarian signal failure,-0.002860,-0.306662,11,-0.032073,0.019017,0,0.533137,1,0.569242,1,Apple stock slips after a big cut for iPhone 1...
2024-10-29,Contrarian signal failure,-0.009285,-0.278343,12,-0.043737,-0.008715,0,0.537237,1,0.564559,1,Apple stock slips after a big cut for iPhone 1...
2024-10-31,Contrarian signal failure,-0.020211,-0.232233,7,0.006950,0.047429,0,0.544199,1,0.556909,1,Apple Intelligence is here. Early users are un...


## Interpretation limit

These cases were selected after observing outcomes and illustrate model behavior. They do not provide an unbiased performance estimate, and some dates can appear in more than one category.